# Model author: synthetic intrinsic Purcell filter

This output-free Notebook reviews the complete `CONVERGING` author UX: general RLGC, a declarative Composite, typed views, a five-variable default optimization recipe, winner Direct response, pump-off HB, and an exact report. The installed scaffold fails explicitly; it cannot produce simulation evidence yet.

In [ ]:
from circuit_model import IPFTarget, build_model, build_response_specs, build_session
from scnsim import ReportSpec, load_q2d_rlgc, units as u

## 1. General RLGC and the reusable Composite

`circuit_library.py` uses the same `RLGC` matrix carrier for one-trace and two-trace sections. Each matrix uses conductor-to-reference voltage and positive head-to-tail current in the extractor's recorded `+z` direction. A real model may replace a manual matrix with `load_q2d_rlgc(path, reference_conductor="ground", conductor_map={"T1": "readout", "T2": "filter"})`. The loader preserves the frozen extraction, direction, and all mutual terms; it performs no frequency interpolation. Imported nonzero off-diagonal R is Direct-only in V1, so the later HB step requires diagonal R. Composite `pin()` handles are external wiring terminals for the parent Plan; `coordinate()` handles are analysis-only and cannot be passed to `plan.net()` or `plan.add_port()`.

In [ ]:
model = build_model()
model.shared_short_length.show()
model.idc_finger_length.show()

## 2. One Plan, two typed views

The shared lineage applies PTC to both floating-qubit probes and then applies the same automatic pair transform for Direct and HB. The response view retains the two Port-promoted node coordinates through their `ElectricNodeRef` handles; their final channel IDs equal the Port IDs only because anonymous-node promotion assigned those IDs. The optimization view retains an unported Public node plus the Composite's `filter_open_tail` `CoordinateRef`; it is valid for Direct quantities but is not a wave-port solve view.

In [ ]:
session = build_session(model, workspace="workspace/synthetic_ipf")
session.response_view
session.optimization_view

## 3. Inspect the model-owned default before execution

The Component Library owns baselines, parameter units, fan-out, and affine calibration support. The circuit-model author separately owns the active variables, finite bounds, quantity objectives, relative weights, and CMA-ES controls below.

In [ ]:
target = IPFTarget(
    readout_frequency=6.2 * u.GHz,
    filter_frequency=7.1 * u.GHz,
    transfer_zero_frequency=6.6 * u.GHz,
    coupling=45.0 * u.MHz,
    combined_linewidth=2.0 * u.MHz,
    response_frequencies=tuple((5.0 + 0.02 * i) * u.GHz for i in range(151)),
)
default_spec = model.build_default_optimization_spec(target)
default_spec.show()
default_spec.variable(model.idc_finger_length).bounds
session.run.explain(session.optimization_view, default_spec).show()

## 4. Optional immutable consumer override

This example deliberately extends only the IDC finger-length bound beyond its affine calibration support. The request-local approval is explicit; the original default remains unchanged and the result cannot be described as calibrated Q2D evidence outside support.

In [ ]:
custom_spec = default_spec.with_variable_overrides(
    bounds={
        model.idc_finger_length: (38.0 * u.um, 76.0 * u.um),
    },
    allow_extrapolation=(model.idc_finger_length,),
)
custom_spec.show()

## 5. Optimize once, then reuse the exact winner

Optimization evaluates all five Direct quantities inside one Julia process. The winner `ParameterSet` is then passed explicitly to independent Direct and pump-off HB response requests. `show()` only presents those materialized Results.

In [ ]:
optimization = session.run.optimize(session.optimization_view, default_spec)
winner = optimization.best.parameters
direct_spec, hb_spec = build_response_specs(target)
direct = session.run.solve(session.response_view, direct_spec, parameters=winner)
hb = session.run.solve(session.response_view, hb_spec, parameters=winner)
report = session.run.build_report(
    ReportSpec(inputs=(optimization, direct, hb.cases["pump_off"]))
)
direct.s.show(magnitude="db")
hb.cases["pump_off"].s.show(magnitude="db")
report.show()

## 6. After a kernel restart

Rebuild the same model/session/spec and call `session.run.resolve(session.optimization_view, default_spec)`. `resolve()` verifies the exact receipt and artifacts; it never executes or searches for a latest result. An extrapolated winner reused in a new solve/evaluate request requires a new request-local `ParameterSet` approval.